# Image tiler: fill a page with as many copies of a picture as possible

Set the parameters in the next cell, then **Run All**. The notebook will:
1. load your image from the Desktop
2. for **every** copy count between `MIN_COPIES` and `MAX_COPIES`, find the largest copy size that fits (mixing normal and 90°-rotated copies when that helps)
3. show a preview of each option, marking the one that covers the most paper
4. save the PDF of the option you pick (`CHOSEN_COPIES`)
4. show a preview and save a print-ready PDF next to the image

The first code cell installs the two libraries it needs (Pillow and ReportLab) into the notebook's own Python.

In [ ]:
%pip install -q pillow reportlab matplotlib

In [ ]:
# ======================= PARAMETERS =======================
IMAGE = "photo.jpg"      # file name on your Desktop, or a full path
PAGE_SIZE = "A4"         # A0-A6, B4, B5, Letter, Legal, Tabloid
MIN_COPIES = 6           # lower bound on the number of copies on the page
MAX_COPIES = 10          # upper bound (set equal to MIN_COPIES for an exact number)

# ---- optional extras ----
MARGIN_MM = 5            # empty border around the page (printers need ~3-5 mm)
GAP_MM = 2               # space between copies (for cutting)
ALLOW_ROTATION = True    # allow 90° rotated copies if that fits more
CUT_LINES = True         # draw thin grey outlines to help cutting
OUTPUT_PDF = None        # None = save next to the image as <name>_<page>.pdf
# ==========================================================

In [ ]:
import math
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

PAGE_SIZES_MM = {
    "A0": (841, 1189), "A1": (594, 841), "A2": (420, 594), "A3": (297, 420),
    "A4": (210, 297), "A5": (148, 210), "A6": (105, 148),
    "B4": (250, 353), "B5": (176, 250),
    "LETTER": (215.9, 279.4), "LEGAL": (215.9, 355.6), "TABLOID": (279.4, 431.8),
}

def find_image(name):
    p = Path(name).expanduser()
    if p.is_file():
        return p
    for desk in [Path.home() / "Desktop", Path.home() / "Scrivania", Path.home() / "OneDrive" / "Desktop"]:
        if (desk / name).is_file():
            return desk / name
    raise FileNotFoundError(f"Could not find '{name}' on your Desktop or as a path.")

img_path = find_image(IMAGE)
img = ImageOps.exif_transpose(Image.open(img_path))   # respect phone-camera rotation
px_w, px_h = img.size

assert 1 <= MIN_COPIES <= MAX_COPIES, "Need 1 <= MIN_COPIES <= MAX_COPIES"
aspect = px_h / px_w     # copies keep the image's proportions
page_w, page_h = PAGE_SIZES_MM[PAGE_SIZE.upper()]
print(f"Image: {img_path.name}  ({px_w}x{px_h} px)")
print(f"Page {PAGE_SIZE}: {page_w} x {page_h} mm   |   copies wanted: {MIN_COPIES}-{MAX_COPIES}")

## Size & packing
For a given copy size, the packer tries every guillotine-style split: a block of copies in one orientation with the leftover strip filled (recursively) in the other orientation. A binary search then finds the largest size at which each copy count fits, for every count in your range.

In [ ]:
from functools import lru_cache

g = GAP_MM
W = page_w - 2 * MARGIN_MM
H = page_h - 2 * MARGIN_MM

def fit(length, size):
    """How many items of `size` fit in `length` with gaps between them."""
    return max(0, int((length + g + 1e-9) // (size + g)))

def pack(item_w, item_h):
    """Best layout for copies of item_w x item_h mm. Returns (count, [(x, y, w, h, rotated), ...])."""
    orients = [(item_w, item_h, False)]
    if ALLOW_ROTATION and abs(item_w - item_h) > 1e-9:
        orients.append((item_h, item_w, True))

    @lru_cache(maxsize=None)
    def best(Wi, Hi, depth):
        Wr, Hr = Wi / 1000, Hi / 1000          # integer microns -> mm (for caching)
        best_n, best_lay = 0, []
        for (w, h, rot) in orients:
            cols, rows = fit(Wr, w), fit(Hr, h)
            if cols * rows > best_n:            # plain grid
                best_n = cols * rows
                best_lay = [(c * (w + g), r * (h + g), w, h, rot) for r in range(rows) for c in range(cols)]
            if depth == 0 or cols == 0 or rows == 0:
                continue
            for k in range(1, rows):            # k rows on top + recurse below
                n2, lay2 = best(Wi, round((Hr - k * (h + g)) * 1000), depth - 1)
                if k * cols + n2 > best_n:
                    best_n = k * cols + n2
                    best_lay = [(c * (w + g), r * (h + g), w, h, rot) for r in range(k) for c in range(cols)] + \
                               [(x, y + k * (h + g), ww, hh, rr) for (x, y, ww, hh, rr) in lay2]
            for k in range(1, cols):            # k columns on the left + recurse right
                n2, lay2 = best(round((Wr - k * (w + g)) * 1000), Hi, depth - 1)
                if k * rows + n2 > best_n:
                    best_n = k * rows + n2
                    best_lay = [(c * (w + g), r * (h + g), w, h, rot) for c in range(k) for r in range(rows)] + \
                               [(x + k * (w + g), y, ww, hh, rr) for (x, y, ww, hh, rr) in lay2]
        return best_n, best_lay

    return best(round(W * 1000), round(H * 1000), 3)

def largest_width_for(n):
    """Largest copy width (mm) at which at least n copies fit."""
    lo, hi = 0.0, max(W, H / aspect)
    for _ in range(40):
        mid = (lo + hi) / 2
        if pack(mid, mid * aspect)[0] >= n:
            lo = mid
        else:
            hi = mid
    return lo

def centred(layout):
    """Shift a layout (relative to the usable area) so it sits centred on the page."""
    max_x = max(x + w for x, y, w, h, r in layout)
    max_y = max(y + h for x, y, w, h, r in layout)
    off_x = MARGIN_MM + (W - max_x) / 2
    off_y = MARGIN_MM + (H - max_y) / 2
    return [(x + off_x, y + off_y, w, h, r) for x, y, w, h, r in layout]   # mm, origin top-left

# one option per copy count: the biggest copies that still fit n of them
options = {}
for n in range(MIN_COPIES, MAX_COPIES + 1):
    w = largest_width_for(n)
    if w <= 0:
        print(f"{n} copies: does not fit")
        continue
    cnt, lay = pack(w, w * aspect)
    placements = centred(lay[:n])
    options[n] = dict(
        n=n, w=w, h=w * aspect, placements=placements,
        rotated=sum(r for *_, r in placements),
        coverage=n * w * w * aspect / (page_w * page_h) * 100,
        dpi=px_w / (w / 25.4),
    )

if not options:
    raise ValueError("No copy count in the range fits - reduce MARGIN_MM / GAP_MM or the copy counts.")

best_n = max(options, key=lambda k: (round(options[k]["coverage"], 6), k))
print(f"{'copies':>6} {'size (mm)':>15} {'upright/rot':>12} {'coverage':>9} {'dpi':>6}")
for n, o in options.items():
    flag = "  <- best coverage" if n == best_n else ""
    soft = " (soft)" if o["dpi"] < 150 else ""
    print(f"{n:>6} {o['w']:>6.1f} x {o['h']:<6.1f} {n - o['rotated']:>5}/{o['rotated']:<5} {o['coverage']:>8.0f}% {o['dpi']:>6.0f}{soft}{flag}")

## Previews - one per copy count

In [ ]:
import matplotlib.pyplot as plt

def render_page(opt, dpi=60):
    mm2px = dpi / 25.4
    page = Image.new("RGB", (round(page_w * mm2px), round(page_h * mm2px)), "white")
    draw = ImageDraw.Draw(page)
    rgb = img.convert("RGB")
    rgb.thumbnail((800, 800))                      # fast previews
    rot = rgb.rotate(90, expand=True)
    for x, y, w, h, r in opt["placements"]:
        tile = (rot if r else rgb).resize((max(1, round(w * mm2px)), max(1, round(h * mm2px))))
        box = (round(x * mm2px), round(y * mm2px))
        page.paste(tile, box)
        if CUT_LINES:
            draw.rectangle([box[0], box[1], box[0] + tile.width, box[1] + tile.height], outline=(170, 170, 170))
    return page

COLS = 4
n_opts = len(options)
rows = math.ceil(n_opts / COLS)
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 3.6, rows * 3.6 * page_h / page_w + rows * 0.4), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, (n, o) in zip(axes.flat, options.items()):
    ax.imshow(render_page(o))
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False,
                               lw=3 if n == best_n else 0.8, ec="#d62728" if n == best_n else "#888"))
    star = "  ★ best" if n == best_n else ""
    ax.set_title(f"{n} copies{star}\n{o['w']:.0f}x{o['h']:.0f} mm · {o['coverage']:.0f}% · {o['dpi']:.0f} dpi", fontsize=9,
                 color="#d62728" if n == best_n else "black")
plt.tight_layout()
plt.show()

## Pick one
Set `CHOSEN_COPIES` to the number under the preview you like (leave `None` for the ★ best-coverage option), then run this cell and the next one. Set `SAVE_ALL` to `True` to get a PDF for every option instead.

In [ ]:
CHOSEN_COPIES = None     # e.g. 8 ; None = best coverage
SAVE_ALL = False

chosen = CHOSEN_COPIES or best_n
if chosen not in options:
    raise ValueError(f"{chosen} is not one of the previewed options: {list(options)}")
o = options[chosen]
print(f"Chosen: {chosen} copies of {o['w']:.1f} x {o['h']:.1f} mm")
display(ImageOps.expand(render_page(o, dpi=90), border=1, fill=(120, 120, 120)))

## Save the PDF (full resolution)

In [ ]:
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.lib.units import mm
from reportlab.lib.utils import ImageReader

src = img if img.mode in ("RGB", "RGBA", "L") else img.convert("RGB")
reader_up = ImageReader(src)
reader_rot = ImageReader(src.rotate(90, expand=True))

def save_pdf(opt, path):
    c = rl_canvas.Canvas(str(path), pagesize=(page_w * mm, page_h * mm))
    c.setTitle(f"{img_path.name} x{opt['n']} on {PAGE_SIZE}")
    for x, y, w, h, rot in opt["placements"]:
        y_pdf = page_h - y - h                  # PDF origin is bottom-left
        c.drawImage(reader_rot if rot else reader_up, x * mm, y_pdf * mm, w * mm, h * mm, mask="auto")
        if CUT_LINES:
            c.setStrokeColorRGB(0.7, 0.7, 0.7)
            c.setLineWidth(0.25)
            c.rect(x * mm, y_pdf * mm, w * mm, h * mm)
    c.save()
    print(f"Saved: {path}")

def default_path(n):
    return img_path.with_name(f"{img_path.stem}_{PAGE_SIZE.upper()}_x{n}.pdf")

if SAVE_ALL:
    for n, opt in options.items():
        save_pdf(opt, default_path(n))
else:
    save_pdf(options[chosen], Path(OUTPUT_PDF).expanduser() if OUTPUT_PDF else default_path(chosen))